In [18]:
import pandas as pd
import re
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [19]:
df = pd.read_csv('test_results.csv')

In [20]:
df.head()

,Format,sample,Si_Al,V_total,V_micro,V_meso,Bronsted_Acid_Sites,Lewis_Acid_Sites,S_BET,S_ext,D_micro,D_meso,crystallinity,metal_content,doi,source,Unnamed: 16
0,Truth,Beta,NaN,0.27 cm³/g,0.22 cm³/g,0.05 cm³/g,NaN,NaN,475 m²/g,NaN,0.55 nm,–,NaN,NaN,NaN,real,NaN
1,Single-Head,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,flat_table,NaN
2,String,Beta,NaN,0.27 cm³/g - t-plot,0.05 cm³/g,NaN,0.24 mol - IR | 0.05 mol - TP,NaN,0.37 m²/g - BET,NaN,0.55 nm,NaN,NaN,NaN,NaN,string,NaN
3,Full-Table,Beta,NaN,0.27 cm³/g - t-plot,0.22 cm³/g - t-plot,0.05 cm³/g - t-plot,NaN,NaN,475 m²/g,NaN,NaN,0.55 nm,NaN,NaN,NaN,table,NaN
4,Truth,MBeta,NaN,0.42 cm³/g,0.18 cm³/g,0.24 cm³/g,NaN,NaN,550 m²/g,NaN,0.58 nm,5.20 nm,NaN,NaN,NaN,real,NaN


In [21]:
def normalize_value(val, split_pipes=True):
    """Return a set of normalized strings or numbers from the value, split by '|' if needed."""
    if pd.isna(val):
        return set()
    val = val.strip()
    parts = [val] if not split_pipes else val.split('|')
    normalized = set()
    for part in parts:
        part_clean = part.strip().replace(' ', '')
        if part_clean:
            normalized.add(part_clean)
    return normalized

def normalize_numeric(val):
    """Return a set of numeric-only substrings split by '|', ignoring spaces and order."""
    if pd.isna(val):
        return set()
    val = val.replace(' ', '')
    parts = val.split('|')
    values = set()
    for part in parts:
        numbers = re.findall(r'\d+\.?\d*', part)
        if numbers:
            values.add(numbers[0])
    return values

def compute_metrics(gt, pred, value_only=False):
    y_true_sets = []
    y_pred_sets = []

    for gt_val, pred_val in zip(gt, pred):
        if value_only:
            gt_set = normalize_numeric(gt_val)
            pred_set = normalize_numeric(pred_val)
        else:
            gt_set = normalize_value(gt_val, split_pipes=True)
            pred_set = normalize_value(pred_val, split_pipes=True)

        y_true_sets.append(gt_set)
        y_pred_sets.append(pred_set)

    tp = fp = fn = 0
    total = 0
    correct = 0

    for gt_set, pred_set in zip(y_true_sets, y_pred_sets):
        if not gt_set and not pred_set:
            continue  # Skip true negatives

        total += 1
        correct += gt_set == pred_set

        # Partial match logic
        tp += len(gt_set & pred_set)
        fp += len(pred_set - gt_set)
        fn += len(gt_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    accuracy = correct / total if total else 0.0

    return precision, recall, f1, accuracy

In [22]:
formats = ['Single-Head', 'String', 'Full-Table']
ground_truth_rows = df[df['Format'] == 'Truth']
results = []
for fmt in formats:
    pred_rows = df[df['Format'] == fmt]
    for value_only in [False, True]:
        all_gt = []
        all_pred = []
        for col in df.columns[2:]:  # Skip Format and sample
            all_gt.extend(ground_truth_rows[col].tolist())
            all_pred.extend(pred_rows[col].tolist())
        p, r, f1, acc = compute_metrics(all_gt, all_pred, value_only=value_only)
        results.append({
            'Format': fmt,
            'Metric Type': 'Value Match' if value_only else 'Exact Match',
            'Precision': p,
            'Recall': r,
            'F1 Score': f1,
            'Accuracy': acc
        })

results_df = pd.DataFrame(results)

In [23]:
results_df

,Format,Metric Type,Precision,Recall,F1 Score,Accuracy
0,Single-Head,Exact Match,0.323529,0.272727,0.295964,0.268293
1,Single-Head,Value Match,0.755814,0.631068,0.687831,0.580952
2,String,Exact Match,0.280374,0.247934,0.263158,0.252101
3,String,Value Match,0.681319,0.601942,0.639175,0.568627
4,Full-Table,Exact Match,0.387931,0.371901,0.379747,0.381356
5,Full-Table,Value Match,0.860000,0.834951,0.847291,0.851485
